# Tutorial 9: Exam Preparation — Classification, Interpretability, and Neural Networks

**Big Data in Finance** | Imperial College Business School  
**Duration:** 1 hour

---

## Purpose

This tutorial continues the **Section B (Applied Problems)** preparation from Tutorial 8, now covering the classification and interpretability side of the course.

### How to Use This Tutorial

1. **Try each exercise on paper first** — this is how you'll work in the exam
2. Then use the code cells to **verify your answers**
3. Solutions are provided at the end of each exercise — try before you peek!

### Coverage

- Exercise 1: Confusion matrix analysis with asymmetric costs
- Exercise 2: Debugging a credit risk classification pipeline
- Exercise 3: Interpreting SHAP output
- Exercise 4: Neural networks — architecture, scaling, and regularization

---

## Setup

Run the cell below to load libraries.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

---

## Exercise 1: Confusion Matrix and Threshold Tuning [~15 minutes]

You have built a Gradient Boosting classifier to predict loan defaults for a bank. The model is evaluated on a hold-out test set of 2,000 loans. Below are the confusion matrices at **two different classification thresholds**.

### Threshold = 0.5 (default)

| | Predicted: No Default | Predicted: Default |
|---|:---:|:---:|
| **Actual: No Default** | 1,750 | 30 |
| **Actual: Default** | 180 | 40 |

### Threshold = 0.3 (lowered)

| | Predicted: No Default | Predicted: Default |
|---|:---:|:---:|
| **Actual: No Default** | 1,640 | 140 |
| **Actual: Default** | 80 | 140 |

**The bank estimates that:**
- Each missed default (false negative) costs **£40,000** in losses
- Each incorrectly rejected good applicant (false positive) costs **£2,000** in lost business

### Questions

1. For **each threshold**, calculate: accuracy, precision (default class), recall (default class), and F1-score.

2. For **each threshold**, calculate the **total cost of errors**.

3. The bank's risk officer says: "We should use threshold 0.5 because it has higher accuracy." Do you agree? Explain.

4. Under what circumstances might you want to lower the threshold even further (e.g., to 0.2)? What is the trade-off?

### ✍️ Your Answer

**1. Metrics:**

| Metric | Threshold = 0.5 | Threshold = 0.3 |
|--------|:---:|:---:|
| Accuracy | ... | ... |
| Precision | ... | ... |
| Recall | ... | ... |
| F1-Score | ... | ... |

**2. Total cost:**
- Threshold 0.5: ...
- Threshold 0.3: ...

**3.** ...

**4.** ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**1. Metrics:**

**Threshold = 0.5:**
- TP = 40, TN = 1750, FP = 30, FN = 180
- Accuracy = (1750 + 40) / 2000 = **89.5%**
- Precision = 40 / (40 + 30) = 40/70 ≈ **57.1%**
- Recall = 40 / (40 + 180) = 40/220 ≈ **18.2%**
- F1 = 2 × (0.571 × 0.182) / (0.571 + 0.182) ≈ **27.6%**

**Threshold = 0.3:**
- TP = 140, TN = 1640, FP = 140, FN = 80
- Accuracy = (1640 + 140) / 2000 = **89.0%**
- Precision = 140 / (140 + 140) = 140/280 = **50.0%**
- Recall = 140 / (140 + 80) = 140/220 ≈ **63.6%**
- F1 = 2 × (0.500 × 0.636) / (0.500 + 0.636) ≈ **56.0%**

**2. Total cost of errors:**

**Threshold = 0.5:**
- FN cost: 180 × £40,000 = £7,200,000
- FP cost: 30 × £2,000 = £60,000
- **Total: £7,260,000**

**Threshold = 0.3:**
- FN cost: 80 × £40,000 = £3,200,000
- FP cost: 140 × £2,000 = £280,000
- **Total: £3,480,000**

**3.** The risk officer is **wrong**. While the accuracy is nearly identical (89.5% vs 89.0%), the total cost differs dramatically: **£7.26M vs £3.48M** — the lower threshold saves the bank almost £3.8M. Accuracy is misleading here because the classes are imbalanced (only 220/2000 = 11% are defaults). The 0.5 threshold misses 180 out of 220 defaults (82% of all defaults go undetected!). The slight accuracy drop at threshold 0.3 comes from rejecting more good applicants (FP increases from 30 to 140), but this is a much cheaper error than missing a default. When error costs are asymmetric, accuracy is the wrong metric.

**4.** Lowering the threshold further (e.g., to 0.2) would catch even more defaults (higher recall) at the cost of rejecting more good applicants (lower precision, more false positives). This makes sense when: (i) the cost of missed defaults is very high relative to lost business, (ii) the bank is in a conservative risk management phase, or (iii) regulatory requirements demand high default detection rates. The trade-off is that you reject more creditworthy borrowers, reducing lending volume and revenue. At the extreme (threshold → 0), you reject everyone, catching all defaults but losing all good business.

</details>

In [ ]:
# Verify your calculations

def compute_metrics(TP, TN, FP, FN, cost_fn, cost_fp):
    total = TP + TN + FP + FN
    accuracy = (TP + TN) / total
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    total_cost = FN * cost_fn + FP * cost_fp
    
    print(f"Accuracy:  {accuracy:.1%}")
    print(f"Precision: {precision:.1%}")
    print(f"Recall:    {recall:.1%}")
    print(f"F1-Score:  {f1:.1%}")
    print(f"Total cost: £{total_cost:,.0f}")

print("=== Threshold 0.5 ===")
compute_metrics(TP=40, TN=1750, FP=30, FN=180, cost_fn=40000, cost_fp=2000)

print("\n=== Threshold 0.3 ===")
compute_metrics(TP=140, TN=1640, FP=140, FN=80, cost_fn=40000, cost_fp=2000)

---

## Exercise 2: Debugging a Credit Risk Pipeline [~15 minutes]

A junior data scientist has written the following code to build a credit risk model using Random Forest classification on LendingClub data. The model achieves an AUC of 0.94 on the test set, which seems suspiciously high.

**The code contains THREE errors.** Identify each, explain why it is problematic, and provide the fix.

```python
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load data
data = pd.read_csv('lending_club.csv')

# Features and target
features = ['loan_amnt', 'annual_inc', 'dti', 'fico_score',
            'revol_util', 'total_pymnt', 'recoveries',          # Line A
            'emp_length', 'home_ownership_encoded']
X = data[features]
y = data['default']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)                      # Line B

# Fit model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,                                             # Line C
    random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC: {auc:.4f}')
```

### ✍️ Your Answer

**Mistake 1 (Line __):**
- Problem: ...
- Fix: ...

**Mistake 2 (Line __):**
- Problem: ...
- Fix: ...

**Mistake 3 (Line __):**
- Problem: ...
- Fix: ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**Mistake 1 — Line A: Feature leakage through `total_pymnt` and `recoveries`**

- **Problem:** `total_pymnt` (total payments received) and `recoveries` (amount recovered after default) are features that are only known *after* the loan outcome is determined. If a borrower defaulted, `recoveries` will be non-zero; if they didn't default, `total_pymnt` will equal the full loan amount. These features essentially encode the target variable. This is **look-ahead bias** at the feature level — at the time of the lending decision, you would not have this information.
- **Fix:** Remove these features and use only information available at the time of loan origination:
```python
features = ['loan_amnt', 'annual_inc', 'dti', 'fico_score',
            'revol_util', 'emp_length', 'home_ownership_encoded']
```

**Mistake 2 — Line B: Random split instead of temporal split**

- **Problem:** `train_test_split` with default settings randomly shuffles the data. For loan data with a temporal structure, this means the model trains on loans issued in 2019 to predict defaults on loans from 2017. This creates look-ahead bias and produces overly optimistic performance. Credit conditions change over time (e.g., 2008 crisis), so random splitting overstates how well the model generalises to future loans.
- **Fix:** Split chronologically:
```python
# Sort by issue date, then split
data = data.sort_values('issue_date')
split_idx = int(0.8 * len(data))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
```

**Mistake 3 — Line C: No class imbalance handling and unlimited tree depth**

- **Problem:** Two issues. First, `max_depth=None` allows trees to grow fully, which risks overfitting — each tree memorises the training data. Second, and more importantly, there is **no handling of class imbalance**. If only ~3% of loans default, the model will overwhelmingly predict "no default" because that minimises the overall error rate. The AUC may still look decent if a few strong features (especially the leaked ones) separate the classes, but the model is not properly calibrated.
- **Fix:**
```python
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,                        # Limit tree depth
    class_weight='balanced',             # Handle class imbalance
    random_state=42)
```

**Note:** The most critical error is Line A (feature leakage). This alone could explain the 0.94 AUC — `recoveries` and `total_pymnt` almost perfectly reveal whether a loan defaulted.

</details>

---

## Exercise 3: Interpreting SHAP Output [~15 minutes]

You have trained an XGBoost classifier for credit default prediction and generated SHAP explanations. Below are two types of SHAP output.

### Part (a): SHAP Summary Data

The following table shows the **mean absolute SHAP values** (global feature importance) across all test observations:

| Feature | Mean |SHAP| | Direction |
|---------|:---:|---|
| fico_score | 0.082 | Higher score → lower default probability |
| dti | 0.065 | Higher DTI → higher default probability |
| annual_inc | 0.041 | Higher income → lower default probability |
| revol_util | 0.038 | Higher utilization → higher default probability |
| loan_amnt | 0.022 | Larger loan → slightly higher default probability |
| emp_length | 0.011 | Longer employment → slightly lower default probability |
| home_ownership | 0.008 | Own > Rent > Mortgage (for default risk) |

**Questions:**

1. What is the difference between SHAP feature importance and the traditional Gini importance from Random Forest? Why might they give different rankings?

2. A regulator asks: "Is your model discriminating based on income?" How would you use SHAP to respond to this concern?

### Part (b): SHAP Waterfall for a Specific Applicant

A loan application was **rejected** (predicted default probability = 0.72). The SHAP waterfall shows the following contributions:

| Feature | Value | SHAP Contribution |
|---------|-------|:---:|
| Base value (population average) | — | 0.12 |
| fico_score = 590 | Low | +0.28 |
| dti = 35.2 | High | +0.18 |
| revol_util = 89% | Very high | +0.11 |
| annual_inc = £85,000 | Above average | -0.06 |
| loan_amnt = £15,000 | Moderate | +0.04 |
| emp_length = 8 years | Long | -0.03 |
| home_ownership = Rent | — | +0.08 |
| **Final prediction** | | **0.72** |

**Questions:**

3. Verify that the SHAP values sum correctly. Show your calculation.

4. The applicant asks: "What can I do to get approved?" Based on the SHAP values, which factors are most within the applicant's control to change, and which would have the largest impact on the prediction?

5. Explain the difference between **global** and **local** interpretability. Which type does the summary table (Part a) provide, and which does the waterfall (Part b) provide?

### ✍️ Your Answer

1. ...
2. ...
3. ...
4. ...
5. ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**1.** Gini importance measures how much a feature reduces impurity (Gini index) when used for splits, averaged across all trees. SHAP values measure each feature's marginal contribution to the prediction, based on Shapley values from cooperative game theory.

They can give different rankings because: (i) Gini importance is biased toward features with many unique values (continuous features get more opportunities to split), while SHAP corrects for this; (ii) Gini importance doesn't account for the direction of the effect (only that the feature was useful), while SHAP shows positive and negative contributions; (iii) correlated features can split Gini importance between them, while SHAP assigns importance based on marginal contribution.

**2.** SHAP provides a principled way to address this concern. You could show that: (i) `annual_inc` has a mean |SHAP| of 0.041, making it the third most important feature — present but not dominant; (ii) the direction makes economic sense (higher income → lower default probability), which is a legitimate creditworthiness signal, not discrimination; (iii) for individual rejections, you can show the SHAP waterfall to demonstrate whether income was the deciding factor or whether other features (like FICO score or DTI) drove the prediction. If income has a large negative SHAP value for a rejected applicant (as in Part b: -0.06), that means income was actually *helping* the applicant — the rejection was driven by other factors.

**3.** SHAP local accuracy property: the base value plus all SHAP contributions must equal the final prediction.

0.12 + 0.28 + 0.18 + 0.11 + (−0.06) + 0.04 + (−0.03) + 0.08 = **0.72** ✓

This is the **local accuracy** (or **efficiency**) property of Shapley values: $f(x) = \phi_0 + \sum_{j} \phi_j$.

**4.** The two largest drivers of the rejection are:
- **FICO score = 590** (SHAP: +0.28, pushing toward default)
- **DTI = 35.2** (SHAP: +0.18, pushing toward default)

Actionable advice:
- **DTI (debt-to-income):** The applicant could reduce existing debt before reapplying. Reducing DTI from 35% to below 20% would substantially reduce this contribution. This is moderately within their control.
- **Revolving utilization (89%):** Paying down credit card balances to reduce utilization below 30% would reduce the +0.11 contribution. This is directly within their control and relatively quick to address.
- **FICO score:** Improving the FICO score from 590 to 650+ would have the largest impact (+0.28 contribution) but takes longer — it requires consistent on-time payments and reduced utilization over several months.

Note that annual_inc (-0.06) and emp_length (-0.03) are already helping the applicant. Home ownership (+0.08) is harder to change in the short term.

**5.** **Global interpretability** answers "How does the model work on average across all predictions?" — it shows which features matter most overall and their general direction of effect. The SHAP summary table in Part (a) provides global interpretability.

**Local interpretability** answers "Why did the model make *this specific prediction* for *this specific observation*?" — it shows how each feature pushed the prediction for one individual. The SHAP waterfall in Part (b) provides local interpretability.

Both are important: global interpretability helps you understand and validate the model overall; local interpretability helps you explain individual decisions to applicants and regulators.

</details>

---

## Exercise 4: Neural Networks [~15 minutes]

### Part (a): Code Debugging

The following code trains a neural network for market return prediction. It contains **two errors**. Identify and fix them.

```python
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit

# Data already split: X_train, X_test, y_train, y_test

# Fit neural network
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.01,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.2,
    random_state=42
)

mlp.fit(X_train, y_train)                        # Line A

# Predict and evaluate
y_pred = mlp.predict(X_test)
oos_r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum(y_test**2)  # Line B
print(f'OOS R²: {oos_r2:.4f}')
```

### Part (b): Conceptual Questions

1. Why is feature scaling essential for neural networks but not for tree-based methods (Random Forest, XGBoost)?

2. The code uses `early_stopping=True`. Explain what this does and how it relates to the bias-variance tradeoff.

3. Your colleague says: "Neural networks are universal approximators, so they should always outperform Ridge regression." Is this correct? Explain, with reference to the signal-to-noise ratio in financial data.

### ✍️ Your Answer

**Part (a):**

- Error 1 (Line __): ...
- Error 2 (Line __): ...

**Part (b):**

1. ...
2. ...
3. ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**Part (a):**

**Error 1 — Line A: No feature scaling before fitting the neural network**

- **Problem:** The neural network is trained on unscaled features. Neural networks use gradient descent to optimise weights, and gradient updates depend on the magnitude of input features. If features have very different scales (e.g., income in thousands vs. ratios between 0 and 1), the loss surface becomes elongated and gradient descent converges slowly or gets stuck. This leads to poor performance or failure to converge.
- **Fix:** Scale features before training:
```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

mlp.fit(X_train_scaled, y_train)
y_pred = mlp.predict(X_test_scaled)
```

**Error 2 — Line B: Wrong OOS R² formula**

- **Problem:** The denominator is `np.sum(y_test**2)`, which computes $\sum y_t^2$ — the sum of squared test values around zero. The correct OOS R² denominator should use the training set mean as the benchmark: $\sum(y_t - \bar{y}_{\text{train}})^2$. Using $\sum y_t^2$ implicitly benchmarks against a prediction of zero, which is only correct if the training mean is exactly zero.
- **Fix:**
```python
train_mean = y_train.mean()
oos_r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - train_mean)**2)
```

**Part (b):**

**1.** Neural networks learn weights via gradient descent. The gradient with respect to a weight is proportional to the corresponding input feature's magnitude. If one feature ranges from 0 to 100,000 (e.g., income) and another from 0 to 1 (e.g., a ratio), the income feature will dominate the gradient updates, making it very hard for the network to learn the importance of smaller-scale features. Standardising all features to mean 0, standard deviation 1 creates a smoother loss surface where gradient descent works efficiently.

Tree-based methods don't need scaling because they use **threshold-based splits** ("is feature X > some value?"). These splits are invariant to monotonic transformations of the features — multiplying a feature by 1000 changes the split threshold value but not which observations go left or right.

**2.** Early stopping monitors the validation loss during training and stops when it starts increasing (after `n_iter_no_change` consecutive epochs of no improvement). In terms of bias-variance:
- With **few epochs** (early stop): higher bias, lower variance (underfitting — the model hasn't learned enough)
- With **many epochs** (no early stop): lower bias, higher variance (overfitting — the model memorises training data)
- Early stopping finds the **sweet spot** where validation error is minimised — the number of training epochs acts as a regularisation parameter, similar to how λ controls complexity in Ridge.

**3.** This is **incorrect**. The universal approximation theorem says a neural network *can* approximate any function given enough neurons and data. It does not say it *will* in practice. Three reasons why Ridge can beat neural networks in finance:

- **Low signal-to-noise ratio:** Financial returns are dominated by noise. A flexible model like a neural network can easily fit the noise rather than the signal, leading to overfitting. Ridge's strong regularisation constrains the model to find only the most robust linear patterns.
- **Small sample size:** Monthly financial time series have hundreds of observations, not millions. Neural networks are data-hungry — they need large samples to estimate their many parameters reliably.
- **Bias-variance tradeoff:** The added flexibility of neural networks (lower bias) comes at the cost of higher variance. In noisy, short-sample settings, the variance increase outweighs the bias reduction, and the simpler Ridge model wins.

"Can approximate" ≠ "Will approximate well in practice."

</details>

---

## 🎯 Key Takeaways for the Exam

1. **Confusion matrix arithmetic:** practice computing precision, recall, F1, and costs by hand — it's mechanical but easy to mess up under time pressure
2. **Threshold tuning** trades off precision vs. recall; the right threshold depends on the **asymmetric costs** of errors
3. **Feature leakage** in classification can come from features that are only known *after* the outcome (e.g., `recoveries`)
4. **SHAP values must sum** to the prediction minus the base value — check this to verify your understanding
5. **Global vs. local interpretability:** summary plots = global, waterfall plots = local
6. **Neural networks require scaling** because gradient descent is sensitive to feature magnitudes; trees do not
7. **Complexity ≠ better performance:** in low signal-to-noise environments, simpler models often win

---

## 📋 Exam Checklist: Quick Reference

Use this to do a final review before the exam:

| Topic | Can you...? |
|-------|-------------|
| Bias-variance | Explain the decomposition and apply it to any method? |
| Ridge vs Lasso | Explain when to prefer each? Write the objective function? |
| Trees vs Forests | Explain how bagging + random features reduce variance? |
| Boosting | Explain sequential fitting to residuals? Compare with bagging? |
| Time-series CV | Explain why K-Fold fails? Describe TimeSeriesSplit? |
| OOS R² | Write the formula with training mean? Interpret negative values? |
| Logistic regression | Write the sigmoid? Interpret coefficients as log-odds? |
| Confusion matrix | Compute accuracy, precision, recall, F1 from a table? |
| Class imbalance | Name two techniques and explain the mechanism? |
| AUC-ROC | Explain the probabilistic interpretation? |
| PCA | Explain variance explained? Fit-on-train principle? |
| SHAP | Verify the sum property? Distinguish global vs local? |
| Neural networks | Explain why scaling is needed? What early stopping does? |
| Data leakage | Spot three types in a code snippet? |
| Statistical vs economic | Explain why low R² can be economically valuable? |